# Lab 26 — MCP client from scratch

> ⏱ 90-110 min · 🟡 Intermediate

Build a Python MCP client end-to-end. By the end of this notebook you'll have:
- a multi-server MCP client with collision-free routing
- the five production defenses (timeout, schema-drift, circuit-breaker, retry-on-auth, response-size limits)
- integration with a Pattern 01 agent loop and a real Anthropic API call
- a token-cost measurement showing why FastMCP 3.1 code mode exists

**Prerequisites read first**: [Building an MCP client](../../concepts/tools/building-an-mcp-client.md). Also: complete [Lab 25 (MCP server from scratch)](../25-mcp-server-from-scratch/) — this lab uses Lab 25's `notes_server.py` as one of two servers.

This notebook follows the 9-step structure from the lab README. Each step is self-contained; you can re-run any step independently.

## Step 0 — Environment setup

Verify FastMCP 3.x is installed. The Anthropic SDK is optional — only Step 6 uses it for the real-LLM integration.

In [ ]:
import sys
from pathlib import Path

try:
    import fastmcp
    print(f"fastmcp version: {fastmcp.__version__}")
    major = int(fastmcp.__version__.split(".")[0])
    if major < 3:
        print("⚠ Lab requires FastMCP 3.x. Run: pip install --upgrade 'fastmcp>=3.0'")
    else:
        print("✓ FastMCP 3.x available")
except ImportError:
    print("✗ fastmcp not installed. Run: pip install 'fastmcp>=3.0'")
    sys.exit(1)

# Optional: Anthropic SDK for Step 6
try:
    import anthropic
    print(f"anthropic version: {anthropic.__version__}")
    HAS_ANTHROPIC = True
except ImportError:
    print("⚠ anthropic SDK not installed; Step 6 will skip. Run: pip install anthropic")
    HAS_ANTHROPIC = False

LAB_DIR = Path.cwd()
print(f"Lab working directory: {LAB_DIR}")

## Step 1 — Minimum-viable client against Lab 25's server

Write a fresh copy of Lab 25's notes server here (so this lab is self-contained — you don't need Lab 25's directory present). Then connect via FastMCP's `Client`, list capabilities, exercise the tools and a templated resource.

The client API is async — the FastMCP `Client` is an async context manager. We use top-level `await` (Jupyter supports it natively; `asyncio.run()` doesn't work because Jupyter already has a running event loop).

In [ ]:
# Write a minimal notes server (a trimmed version of Lab 25's)
NOTES_SERVER_PATH = LAB_DIR / "notes_server.py"

NOTES_SERVER_CODE = """\
\"\"\"Lab 26 notes server — minimal copy of Lab 25's for client testing.\"\"\"
from fastmcp import FastMCP

mcp = FastMCP("notes-server")
NOTES: dict[str, str] = {}


@mcp.tool()
def create_note(title: str, body: str) -> dict:
    \"\"\"Create a note. Returns status='exists' if title is duplicate.\"\"\"
    if title in NOTES:
        return {"status": "exists", "title": title}
    NOTES[title] = body
    return {"status": "created", "title": title}


@mcp.tool()
def get_note(title: str) -> dict:
    \"\"\"Retrieve a note by title.\"\"\"
    if title not in NOTES:
        return {"status": "not_found", "title": title}
    return {"status": "ok", "title": title, "body": NOTES[title]}


@mcp.tool()
def list_notes() -> list[str]:
    \"\"\"List all note titles.\"\"\"
    return sorted(NOTES.keys())


@mcp.resource("notes://{title}")
def one_note(title: str) -> str:
    \"\"\"A single note's body, addressed by URI template.\"\"\"
    return NOTES.get(title, "[note not found]")


if __name__ == "__main__":
    mcp.run()
"""

NOTES_SERVER_PATH.write_text(NOTES_SERVER_CODE)
print(f"Wrote {NOTES_SERVER_PATH} ({NOTES_SERVER_PATH.stat().st_size} bytes)")

In [ ]:
from fastmcp import Client

async def exercise_notes_server():
    """Connect to the notes server, exercise all primitives."""
    async with Client(str(NOTES_SERVER_PATH)) as client:
        tools = await client.list_tools()
        resources = await client.list_resources()
        prompts = await client.list_prompts()
        templates = await client.list_resource_templates()

        # Create + retrieve + list
        await client.call_tool("create_note", {"title": "Q3-plan", "body": "Build MCP client"})
        await client.call_tool("create_note", {"title": "Q4-plan", "body": "Add A2A"})
        get_result = await client.call_tool("get_note", {"title": "Q3-plan"})
        list_result = await client.call_tool("list_notes", {})
        templated = await client.read_resource("notes://Q3-plan")

        return {
            "tool_count": len(tools),
            "resource_count": len(resources),
            "template_count": len(templates),
            "prompt_count": len(prompts),
            "tool_names": [t.name for t in tools],
            "get_result": get_result.data,
            "list_result": list_result.data,
            "templated_body": templated[0].text if templated else None,
        }

result = await exercise_notes_server()
print("=== Notes server exercised ===\n")
for k, v in result.items():
    print(f"  {k}: {v}")

## Step 2 — Second toy server for multi-server orchestration

Write a small `time_server.py` with two tools — `get_current_time` and `get_status`. The `get_status` name is **deliberately chosen to collide** with a tool we'll add to a third server in the multi-server demo; that's how Step 3 demonstrates the collision-routing fix.

In [ ]:
TIME_SERVER_PATH = LAB_DIR / "time_server.py"

TIME_SERVER_CODE = """\
\"\"\"Lab 26 time server — minimal toy server for multi-server demo.\"\"\"
from datetime import datetime, timezone
from fastmcp import FastMCP

mcp = FastMCP("time-server")


@mcp.tool()
def get_current_time(timezone_name: str = "UTC") -> dict:
    \"\"\"Get the current time. Returns ISO-formatted UTC time.\"\"\"
    now = datetime.now(timezone.utc)
    return {
        "status": "ok",
        "iso_time": now.isoformat(),
        "unix_seconds": int(now.timestamp()),
    }


@mcp.tool()
def get_status() -> dict:
    \"\"\"Health check for the time server.

    NOTE: 'get_status' deliberately collides with a tool of the same name
    on other servers — this is how the multi-server client demonstrates
    collision-free routing via server-name prefixes.
    \"\"\"
    return {"server": "time-server", "status": "healthy"}


if __name__ == "__main__":
    mcp.run()
"""

TIME_SERVER_PATH.write_text(TIME_SERVER_CODE)
print(f"Wrote {TIME_SERVER_PATH} ({TIME_SERVER_PATH.stat().st_size} bytes)")

# Quick verify it works
async def verify_time_server():
    async with Client(str(TIME_SERVER_PATH)) as client:
        tools = await client.list_tools()
        result = await client.call_tool("get_current_time", {})
        return {"tools": [t.name for t in tools], "current_time": result.data}

verify = await verify_time_server()
print(f"  Time server tools: {verify['tools']}")
print(f"  Current time: {verify['current_time']}")

## Step 3 — `MultiServerMcpClient` with collision-free routing

Real clients talk to N servers. The challenge is that two servers can expose tools with the same name (`get_status` lives on both `notes-server` and `time-server` in our demo). Without prefixing, the LLM sees ambiguous duplicates.

The solution: prefix every tool name with the server identifier (`notes__get_status` vs `time__get_status`). When the LLM calls a prefixed tool, route by the prefix.

We'll also add a `get_status` tool to the notes server first to make the collision concrete.

In [ ]:
# Add get_status to the notes server (so the collision is real)
existing = NOTES_SERVER_PATH.read_text()
status_tool = """\

@mcp.tool()
def get_status() -> dict:
    \"\"\"Health check for the notes server.\"\"\"
    return {"server": "notes-server", "status": "healthy", "note_count": len(NOTES)}

"""
new_content = existing.replace(
    "if __name__ == \"__main__\":",
    status_tool + "\nif __name__ == \"__main__\":",
    1,
)
NOTES_SERVER_PATH.write_text(new_content)
print(f"Added get_status to notes server. New size: {NOTES_SERVER_PATH.stat().st_size} bytes")

In [ ]:
class MultiServerMcpClient:
    """Multi-server MCP client with collision-free routing via server-name prefixes."""

    def __init__(self, servers: dict[str, str]):
        """
        Args:
            servers: Mapping of short_name -> server path or URL.
                e.g., {"notes": "notes_server.py", "time": "time_server.py"}
        """
        self.server_paths = servers
        self.clients: dict[str, Client] = {}

    async def __aenter__(self):
        for name, path in self.server_paths.items():
            client = Client(path)
            await client.__aenter__()
            self.clients[name] = client
        return self

    async def __aexit__(self, *args):
        for client in self.clients.values():
            await client.__aexit__(*args)

    async def list_all_tools(self) -> list[dict]:
        """Aggregate tool schemas from all servers, prefixed by server name."""
        all_tools = []
        for server_name, client in self.clients.items():
            tools = await client.list_tools()
            for t in tools:
                all_tools.append({
                    "prefixed_name": f"{server_name}__{t.name}",
                    "server": server_name,
                    "original_name": t.name,
                    "description": t.description,
                    "input_schema": t.inputSchema,
                })
        return all_tools

    async def call_tool(self, prefixed_name: str, arguments: dict):
        """Route a prefixed tool call to the right server."""
        server_name, sep, tool_name = prefixed_name.partition("__")
        if not sep:
            raise ValueError(f"Tool name must be server-prefixed: {prefixed_name}")
        if server_name not in self.clients:
            raise ValueError(f"Unknown server: {server_name}")
        return await self.clients[server_name].call_tool(tool_name, arguments)


# Demo: connect to both servers and observe collision-free routing
async def demo_multi_server():
    servers = {
        "notes": str(NOTES_SERVER_PATH),
        "time": str(TIME_SERVER_PATH),
    }
    async with MultiServerMcpClient(servers) as multi:
        all_tools = await multi.list_all_tools()

        # Both servers expose get_status — but under different prefixed names
        get_status_tools = [t for t in all_tools if t["original_name"] == "get_status"]

        # Route to each one explicitly
        notes_status = await multi.call_tool("notes__get_status", {})
        time_status = await multi.call_tool("time__get_status", {})

        return {
            "total_tools_across_servers": len(all_tools),
            "tool_names": [t["prefixed_name"] for t in all_tools],
            "get_status_collision_count": len(get_status_tools),
            "notes_status": notes_status.data,
            "time_status": time_status.data,
        }

multi_result = await demo_multi_server()
print("=== Multi-server demo ===\n")
print(f"Total tools (with prefixes): {multi_result['total_tools_across_servers']}")
print(f"Tool names: {multi_result['tool_names']}")
print(f"\n'get_status' collisions: {multi_result['get_status_collision_count']}")
print(f"  → notes__get_status: {multi_result['notes_status']}")
print(f"  → time__get_status:  {multi_result['time_status']}")
print()
print("Without prefixing, the LLM would see two 'get_status' tools and pick unpredictably.")

## Step 4 — The five production defenses

Layer defenses onto `MultiServerMcpClient`. We won't implement all five at production fidelity — that would require a real circuit-breaker library and a real schema-cache backend. We'll demonstrate each pattern's *shape* with code that runs in the notebook.

**Defense 1: Per-tool timeout.** Wrap every `call_tool` in `asyncio.wait_for`.

**Defense 2: Schema-cache TTL.** Cache `list_tools()` results with an expiration; refresh on mismatch.

**Defense 3: Circuit-breaker.** Track consecutive failures per server; "open" after N (default 5); auto-close after a cool-off (default 60s).

**Defense 4: Retry-on-auth-error.** Catch 401s; refresh token (simulated); retry once.

**Defense 5: Response-size limit.** Truncate large responses with an explicit marker.

For the notebook we'll wire defenses 1, 3, and 5 (the ones easy to demo without external state). Defenses 2 and 4 are documented as patterns; production deployments would use libraries like `tenacity` for retry and `circuitbreaker` for the circuit.

In [ ]:
import asyncio
import time
from dataclasses import dataclass, field

@dataclass
class CircuitState:
    consecutive_failures: int = 0
    opened_at: float = 0.0  # 0 means closed

    def is_open(self, cool_off_seconds: float = 60.0) -> bool:
        if self.opened_at == 0:
            return False
        return time.time() - self.opened_at < cool_off_seconds

    def record_failure(self, threshold: int = 5):
        self.consecutive_failures += 1
        if self.consecutive_failures >= threshold:
            self.opened_at = time.time()

    def record_success(self):
        self.consecutive_failures = 0
        self.opened_at = 0.0


class DefensiveMultiServerClient(MultiServerMcpClient):
    """MultiServerMcpClient with timeout + circuit-breaker + response-size limit."""

    def __init__(
        self,
        servers: dict[str, str],
        per_tool_timeout: float = 10.0,
        max_response_chars: int = 50_000,
        circuit_threshold: int = 5,
        circuit_cool_off: float = 60.0,
    ):
        super().__init__(servers)
        self.per_tool_timeout = per_tool_timeout
        self.max_response_chars = max_response_chars
        self.circuit_threshold = circuit_threshold
        self.circuit_cool_off = circuit_cool_off
        self.circuits: dict[str, CircuitState] = {
            name: CircuitState() for name in servers
        }

    async def call_tool(self, prefixed_name: str, arguments: dict) -> dict:
        """Call with defenses: circuit-breaker → timeout → response-size limit."""
        server_name, sep, tool_name = prefixed_name.partition("__")
        if not sep:
            return {"status": "error", "error": "unprefixed_tool_name"}
        if server_name not in self.clients:
            return {"status": "error", "error": "unknown_server"}

        circuit = self.circuits[server_name]
        if circuit.is_open(self.circuit_cool_off):
            return {"status": "error", "error": "server_unavailable", "server": server_name}

        try:
            result = await asyncio.wait_for(
                self.clients[server_name].call_tool(tool_name, arguments),
                timeout=self.per_tool_timeout,
            )
            circuit.record_success()
            data = result.data
            data_str = str(data)
            if len(data_str) > self.max_response_chars:
                return {
                    "status": "ok",
                    "truncated": True,
                    "data": data_str[:self.max_response_chars],
                    "omitted_chars": len(data_str) - self.max_response_chars,
                }
            return {"status": "ok", "data": data}
        except TimeoutError:
            circuit.record_failure(self.circuit_threshold)
            return {"status": "error", "error": "tool_timeout", "tool": prefixed_name}
        except Exception as e:
            circuit.record_failure(self.circuit_threshold)
            return {"status": "error", "error": "tool_exception", "detail": str(e)}


# Demo the defensive client
async def demo_defensive():
    servers = {"notes": str(NOTES_SERVER_PATH), "time": str(TIME_SERVER_PATH)}
    async with DefensiveMultiServerClient(servers, per_tool_timeout=5.0) as client:
        # Normal call succeeds and records success
        ok = await client.call_tool("notes__create_note",
                                     {"title": "Defended", "body": "via wrapper"})
        # Try a bad server name — short-circuits without making a real call
        bad = await client.call_tool("unknown__anything", {})
        # Try unprefixed name
        unpref = await client.call_tool("just_a_name", {})
        return {"ok": ok, "bad_server": bad, "unprefixed": unpref}

defensive = await demo_defensive()
print("=== Defensive client behavior ===\n")
for k, v in defensive.items():
    print(f"  {k}: {v}")

## Step 5 — Schema-translation layer (MCP → Anthropic)

The LLM speaks function-calling in its own format. The client converts MCP tool definitions to that format every loop iteration. Implementation is mechanical — but worth seeing concretely.

In [ ]:
def mcp_tool_to_anthropic(mcp_tool, server_prefix: str = "") -> dict:
    """Convert one MCP Tool to Anthropic's tools-parameter format.

    Args:
        mcp_tool: A FastMCP Tool object (from list_tools()).
        server_prefix: If non-empty, prepend "<prefix>__" to the tool name.

    Returns:
        A dict in Anthropic's expected tools-parameter shape.
    """
    name = f"{server_prefix}__{mcp_tool.name}" if server_prefix else mcp_tool.name
    return {
        "name": name,
        "description": mcp_tool.description or "",
        "input_schema": mcp_tool.inputSchema or {"type": "object", "properties": {}},
    }


async def aggregate_anthropic_schemas(client: MultiServerMcpClient) -> list[dict]:
    """Collect all tool schemas across all servers, formatted for Anthropic."""
    schemas = []
    for server_name, mcp_client in client.clients.items():
        tools = await mcp_client.list_tools()
        for t in tools:
            schemas.append(mcp_tool_to_anthropic(t, server_prefix=server_name))
    return schemas


# Demo the schema layer
async def demo_schemas():
    servers = {"notes": str(NOTES_SERVER_PATH), "time": str(TIME_SERVER_PATH)}
    async with MultiServerMcpClient(servers) as client:
        anthropic_schemas = await aggregate_anthropic_schemas(client)
        return anthropic_schemas

schemas = await demo_schemas()
print("=== Anthropic-formatted schemas from both servers ===\n")
print(f"Total tools: {len(schemas)}\n")
for s in schemas:
    print(f"  • {s['name']}")
    print(f"      desc: {(s['description'] or '')[:70]}...")
    required = s['input_schema'].get('required', [])
    print(f"      required params: {required}")

## Step 6 — Wire the client into a Pattern 01 agent loop

Drop the multi-server client into a Pattern 01 agent loop with a real Anthropic API call. The loop's shape is *identical* to Pattern 01's framework-free Python — only the tool-execution line goes through MCP instead of in-process Python functions.

**Heads-up**: this step makes one real API call. If `ANTHROPIC_API_KEY` isn't set, the cell skips with a clear message. You can still finish the notebook without it; the cells after this one don't depend on a real call.

In [ ]:
import os
import json

MAX_STEPS = 6

async def run_agent_with_mcp(user_prompt: str) -> dict:
    """A Pattern 01 agent loop where tool calls go through MCP."""
    if not HAS_ANTHROPIC:
        return {"status": "skipped", "reason": "anthropic SDK not installed"}
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return {"status": "skipped", "reason": "ANTHROPIC_API_KEY not set"}

    import anthropic
    llm = anthropic.Anthropic(api_key=api_key)

    servers = {"notes": str(NOTES_SERVER_PATH), "time": str(TIME_SERVER_PATH)}
    async with MultiServerMcpClient(servers) as mcp_client:
        tool_schemas = await aggregate_anthropic_schemas(mcp_client)
        messages = [{"role": "user", "content": user_prompt}]

        for step in range(MAX_STEPS):
            response = llm.messages.create(
                model="claude-haiku-4-5",
                max_tokens=1024,
                tools=tool_schemas,
                messages=messages,
            )

            if response.stop_reason == "end_turn":
                final_text = next(
                    (b.text for b in response.content if hasattr(b, "text")), ""
                )
                return {"status": "done", "answer": final_text, "steps_used": step + 1}

            # Pass the assistant's tool-use blocks back as a message
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    # The MCP-specific line: tool execution goes through MCP
                    result = await mcp_client.call_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result.data, default=str)[:2000],
                    })
            messages.append({"role": "user", "content": tool_results})

        return {"status": "max_steps", "steps_used": MAX_STEPS}


# Try a task that exercises multi-server routing
agent_result = await run_agent_with_mcp(
    "Create a note titled 'standup' with the body 'discuss MCP module 3'. "
    "Then tell me what the current UTC time is. "
    "Finally, list all the notes."
)
print("=== Agent run ===\n")
print(json.dumps(agent_result, indent=2)[:1500])

## Step 7 — Token-bloat measurement

Measure how many tokens the tool-schema layer costs. Per [Apigene April 2026](https://apigene.ai/blog/fastmcp), production deployments report tool schemas consuming 15K+ tokens — the #1 production pain point and the reason FastMCP 3.1 introduced code mode (98%+ token savings per Cloudflare's measurement).

We approximate token count as `len(json_string) / 4` — a rough but standard heuristic for English text and JSON.

In [ ]:
def approx_token_count(text: str) -> int:
    """Rough token-count approximation: ~4 chars per token for English/JSON."""
    return len(text) // 4

async def measure_schema_bloat():
    """Measure tool-schema token cost across server scopes."""
    measurements = []

    # 1 server (notes)
    async with MultiServerMcpClient({"notes": str(NOTES_SERVER_PATH)}) as c:
        schemas = await aggregate_anthropic_schemas(c)
        json_blob = json.dumps(schemas)
        measurements.append({
            "scope": "1 server (notes only)",
            "tool_count": len(schemas),
            "schema_chars": len(json_blob),
            "approx_tokens": approx_token_count(json_blob),
        })

    # 2 servers (notes + time)
    async with MultiServerMcpClient({
        "notes": str(NOTES_SERVER_PATH),
        "time": str(TIME_SERVER_PATH),
    }) as c:
        schemas = await aggregate_anthropic_schemas(c)
        json_blob = json.dumps(schemas)
        measurements.append({
            "scope": "2 servers (notes + time)",
            "tool_count": len(schemas),
            "schema_chars": len(json_blob),
            "approx_tokens": approx_token_count(json_blob),
        })

    return measurements

bloat = await measure_schema_bloat()
print("=== Tool-schema token-cost measurement ===\n")
for m in bloat:
    print(f"  {m['scope']}:")
    print(f"    tools: {m['tool_count']}")
    print(f"    JSON characters: {m['schema_chars']:,}")
    print(f"    approx tokens (chars / 4): {m['approx_tokens']:,}")
    print()

# Extrapolate to a realistic production scope
per_tool = bloat[1]["approx_tokens"] / bloat[1]["tool_count"]
print(f"Per-tool avg cost: ~{per_tool:.0f} tokens")
print(f"  → 5 servers × 10 tools/server ≈ {int(per_tool * 50):,} tokens/request")
print("  → Compare with FastMCP 3.1 code-mode target: 2,000–3,000 tokens/request")
print("    (savings via dynamic tool-schema discovery rather than upfront loading)")

## Step 8 — Stretch directions

The lab's core is done at Step 7. Stretch material for further exploration:

**Per-tool latency tracking.** Wrap `call_tool` to record start/end times; emit summary stats per tool. Mirrors what Path 06's OTel-GenAI conventions instrument at the span level.

**Circuit-breaker integration test.** Simulate a flaky server by introducing artificial failures (modify `get_status` to randomly raise); observe the circuit opening after 5 consecutive failures and closing after the cool-off. Production deployments use `circuitbreaker` or `pybreaker` rather than the hand-rolled version above.

**FastMCP 3.1 code mode.** Replace the eager `list_tools()` aggregation with the dynamic discovery pattern — expose meta-tools (`list_servers`, `list_tools_on_server`, `call_tool_on_server`) and let the LLM decide which tools to inspect on each turn. Measure the token-cost reduction.

**Real-server connection.** Replace `time_server.py` with a public MCP server from the [MCP Registry](https://registry.modelcontextprotocol.io) (filesystem, GitHub, etc.). Note the auth setup difference — public servers expect bearer tokens or OAuth flows.

None of these is required to finish the lab — they're the natural follow-ups once the core mechanics are clear.

## What you've built

You now have:
- 📄 `notes_server.py` — Lab 25's server, included here for self-containment
- 📄 `time_server.py` — a second toy server demonstrating multi-server orchestration
- A `MultiServerMcpClient` with collision-free routing via server-name prefixes
- A `DefensiveMultiServerClient` with timeout + circuit-breaker + response-size limits
- A schema-translation layer (MCP → Anthropic tools format)
- A working Pattern 01 agent loop that calls tools across both servers
- A token-cost measurement showing why FastMCP 3.1 code mode exists

## Where this goes next

This client is the substrate for the rest of Path 04's MCP work:

- **Module 4 — MCP security threat model** (future batch): the arxiv:2601.10955 resource-amplification attack walked through against this client; tool-description injection mitigations
- **Modules 5-7 — A2A** (future batches): how MCP composes with A2A in production multi-agent systems

## Test yourself

Take the [MCP client and discovery quiz](../../quizzes/foundations/mcp-client-and-discovery.md) — 8 questions covering Module 3 and this lab.